# 01 · Retrieval pipeline (SciFact smoke test, then TREC-COVID)

This notebook only *calls* the code in `src/`; all logic lives there. Run the cells top to bottom.

| Step | What happens | Output (where) |
|---|---|---|
| Setup | Mount Drive, clone/pull the repo, install Java + packages | `/content/robust-news-retrieval` |
| A3 | Documents → one clean JSONL file | Drive: `ir-project-data/<dataset>/corpus/docs.jsonl` |
| A4 | BM25: build Lucene index, search the queries | Drive: `<dataset>/bm25_index/`; repo: `runs/<dataset>/bm25_*.run` |
| A2 | Score the runs | repo: `results/<dataset>/*.csv` + a table printed below |
| Save | Copy runs and results to Drive so they survive the session | Drive: `ir-project-data/outputs/` |

Dense retrieval (A5) and RRF (A6) cells will be added below when those modules exist.

## Setup
**First time in a session:** run the two cells below. The install cell then asks you to restart (**Runtime → Restart session**). After the restart, run both cells again: the install is skipped the second time.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/robust-news-retrieval'
BRANCH = 'main'

if not os.path.exists(REPO):
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q -b {BRANCH} https://{token}@github.com/HamsiniGururaj/robust-news-retrieval.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
import importlib.util

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'  # Pyserini needs Java 21
if importlib.util.find_spec('pyserini') is None:
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    !pip install -q -r requirements.txt
    print('Installed. Now do Runtime -> Restart session, then run the setup cells again.')
else:
    print('Packages already installed.')

In [ ]:
# Unit tests: should end with 'passed'
!python -m pytest -q

## Part 1 · SciFact smoke test
SciFact is small (5k documents), so this finishes in a couple of minutes and proves the pipeline works before the real dataset.

**What to expect:** the BEIR paper and Pyserini report BM25 nDCG@10 of roughly 0.66–0.68 on SciFact. Ours may be slightly lower because we deliberately turn stemming off. A value far below that (e.g. under 0.5) means something is wrong.

In [ ]:
!python -m src.prepare_corpus --dataset scifact
!python -m src.bm25 index --dataset scifact
!python -m src.bm25 search --dataset scifact
!python -m src.evaluate --dataset scifact --runs 'runs/scifact/*.run'

## Part 2 · TREC-COVID (primary benchmark)
171k documents. Corpus prep takes about a minute; indexing a few minutes.

The `index` step prints a check (`probe_document_frequencies`): **`the` must be > 0** (stopwords kept) and **`studi` must be 0** (no stemming). It stops with an error if not.

Two query sets are searched: `query` (the short keyword queries, our *original* queries) and `text` (the question form, secondary).

In [ ]:
!python -m src.prepare_corpus --dataset trec_covid
!python -m src.bm25 index --dataset trec_covid

In [ ]:
!python -m src.bm25 search --dataset trec_covid
!python -m src.bm25 search --dataset trec_covid --queries queries/trec_covid_text.tsv
!python -m src.evaluate --dataset trec_covid --runs 'runs/trec_covid/*.run'

## Save outputs to Drive
`runs/` and `results/` are inside the cloned repo, which is wiped when the session ends. This copies them to Drive (`ir-project-data/outputs/`). To commit them, download that folder and copy `runs/` and `results/` into your local repo.

In [ ]:
OUT = '/content/drive/MyDrive/ir-project-data/outputs'
!mkdir -p {OUT}
!cp -r runs results {OUT}/
!ls -R {OUT} | head -40